In [1]:
import json, re, itertools
from collections import Counter, defaultdict
from datetime import date

BASE = 'liquipedia_data/clean_data/fortnite'
TODAY = date.today().isoformat()

players_rows = json.load(open(f'{BASE}/players.json', encoding='utf-8'))
tournaments  = json.load(open(f'{BASE}/tournaments.json', encoding='utf-8'))
placements   = json.load(open(f'{BASE}/placements.json', encoding='utf-8'))
orgs_payload = json.load(open(f'{BASE}/orgs.json', encoding='utf-8'))

# ------------------------------------------------------------------ resolve --
# Identical to the helper in the career_path and teammates cells: a placement's
# participant name -> a *playable* players.json pagename, or None.
by_page = {p['pagename'].replace('_', ' '): p['pagename'] for p in players_rows}
by_id = {}
for p in players_rows:
    by_id.setdefault(p['id'], p['pagename'])
tier = {p['pagename']: p['tier'] for p in players_rows}
player_of = {p['pagename']: p for p in players_rows}

def resolve(name):
    page = by_page.get(name) or by_id.get(name)
    return page if page and tier.get(page) != 'unused' else None

PLAYABLE = [p for p in players_rows if p['tier'] != 'unused']
print(f'{len(PLAYABLE):,} playable players, {len(placements):,} placement rows')

# -------------------------------------------------------------- classifiers --
def is_epic(t):
    return any('epic games' in str(o).lower() for o in (t.get('organizers') or []))

def is_lan(t):
    """A LAN worth asking about: offline (or hybrid) and top two tiers."""
    return t['type'] in ('Offline', 'Hybrid') and t['liquipediatier'] in (1, 2)

def is_major(t):
    """Your career_path filter, verbatim."""
    return (
        t['liquipediatier'] == 1
        and t['liquipediatiertype'] is None
        and is_epic(t)
        and (t['startdate'] or '') >= '2019-07-26'
        and not re.search(r'console|mobile|twitch', t['name'], re.I)
    )

def is_fncs_final(t):
    return is_major(t) and 'FNCS' in t['name']

def is_world_cup(t):
    return t['name'].startswith('Fortnite World Cup')

def is_global(t):
    return is_world_cup(t) and 'Finals' in t['name'] or 'Global Championship' in t['name']

# Region label: tournaments.json says "Brazil", players.json says "South
# America". One word for one place, or the per-region boards split in two.
REGION_FIX = {'Brazil': 'South America', 'Japan': 'Asia', 'MENA': 'Middle East',
              'India': 'Asia', 'China': 'Asia', 'Latin America': 'South America',
              'CIS': 'Europe', 'Benelux': 'Europe', 'Turkey': 'Europe',
              'Pakistan': 'Asia'}

def region_of(t):
    r = t.get('region')
    return REGION_FIX.get(r, r)

tour_of = {}
for t in tournaments:
    tour_of.setdefault(t['name'], t)

MAJORS = {t['name'] for t in tournaments if is_major(t)}
LANS   = {t['name'] for t in tournaments if is_lan(t)}
FNCS   = {t['name'] for t in tournaments if is_fncs_final(t)}
WCUP   = {t['name'] for t in tournaments if is_world_cup(t)}
GLOBAL = {t['name'] for t in tournaments if is_global(t)}

print(f'majors {len(MAJORS)}  LANs {len(LANS)}  FNCS finals {len(FNCS)} '
      f'  World Cup {len(WCUP)}  globals {len(GLOBAL)}')

# ------------------------------------------------------------- placed rows --
# One pass, reused by every cell below. `rank` is None for '', 'DNP' and 'DQ';
# a range like '35-36' reads as its best end, same rule as career_path.
def rank_of(raw):
    m = re.match(r'^(\d+)', str(raw or ''))
    return int(m.group(1)) if m else None

PLACED = []
for row in placements:
    t = tour_of.get(row.get('tournament'))
    if not t:
        continue
    r = rank_of(row.get('placement'))
    money = float(row.get('individualprizemoney') or 0)
    pages = [(resolve(p.get('player') or ''), p.get('team')) for p in (row.get('participants') or [])]
    pages = [(pg, tm) for pg, tm in pages if pg]
    if not pages:
        continue
    PLACED.append((t, r, money, pages))

print(f'{len(PLACED):,} placement rows with at least one nameable player')

5,678 playable players, 442,736 placement rows
majors 188  LANs 80  FNCS finals 184   World Cup 78  globals 16
221,246 placement rows with at least one nameable player


In [2]:
OUT = f'{BASE}/facts.json'

# ------------------------------------------------------------------ events --
headline = sorted(
    {n for n in (MAJORS | LANS)},
    key=lambda n: (tour_of[n]['startdate'] or '', n),
)
index_of = {n: i for i, n in enumerate(headline)}

def short_name(t):
    """'FNCS 2025 - Major 3: Europe - Grand Finals' -> 'FNCS 2025 - Major 3: Europe'."""
    s = re.sub(r'\s*-?\s*Grand Finals?\s*:?\s*', ' ', t['name'], flags=re.I)
    return re.sub(r'\s{2,}', ' ', s).strip()

events = []
for n in headline:
    t = tour_of[n]
    events.append({
        'name': t['name'],
        'short': short_name(t),
        'date': str(t['startdate'])[:10],
        'kind': ('global' if n in GLOBAL else 'fncs' if n in FNCS
                 else 'lan' if n in LANS else 'major'),
        'lan': n in LANS,
        'region': region_of(t),
        'mode': t['mode'],
        'prizePool': None if t['prizepool'] is None else round(float(t['prizepool'])),
    })

# ----------------------------------------------------------------- players --
blank = lambda: {'played': set(), 'won': set(), 'apps': 0, 'lanApps': 0,
                 'fncsApps': 0, 'winRegions': set(), 'winYears': set()}
facts = defaultdict(blank)

for t, r, money, pages in PLACED:
    name, idx = t['name'], index_of.get(t['name'])
    for page, _team in pages:
        f = facts[page]
        f['apps'] += 1
        if name in LANS:
            f['lanApps'] += 1
        if name in FNCS:
            f['fncsApps'] += 1
        if idx is not None:
            f['played'].add(idx)
            if r == 1:
                f['won'].add(idx)
                reg = region_of(t)
                if reg:
                    f['winRegions'].add(reg)
                f['winYears'].add(int(str(t['startdate'])[:4]))

players_out = []
for page in sorted(facts):
    if tier.get(page) == 'unused':
        continue
    f = facts[page]
    won_kinds = Counter(events[i]['kind'] for i in f['won'])
    players_out.append({
        'id': page,
        'played': sorted(f['played']),
        'won': sorted(f['won']),
        'apps': f['apps'],
        'lanApps': f['lanApps'],
        'fncsApps': f['fncsApps'],
        'wins': {
            'global': won_kinds['global'],
            'fncs': won_kinds['fncs'],
            'lan': sum(1 for i in f['won'] if events[i]['lan']),
            'major': len(f['won']),
        },
        'winRegions': sorted(f['winRegions']),
        'winYears': sorted(f['winYears']),
    })

payload = {'generated': TODAY, 'events': events, 'players': players_out}
with open(OUT, 'w', encoding='utf-8') as fh:
    json.dump(payload, fh, ensure_ascii=False, separators=(',', ':'))

import os
print(f'{len(events)} headline events, {len(players_out):,} players, '
      f'{os.path.getsize(OUT)/1e6:.1f} MB')
print('  LAN winners:      ', sum(1 for p in players_out if p['wins']['lan']))
print('  global winners:   ', sum(1 for p in players_out if p['wins']['global']))
print('  5+ tournaments:   ', sum(1 for p in players_out if p['apps'] >= 5))
print('  most LAN apps:    ', sorted(players_out, key=lambda p: -p['lanApps'])[:3])

259 headline events, 5,677 players, 1.0 MB
  LAN winners:       197
  global winners:    15
  5+ tournaments:    5455
  most LAN apps:     [{'id': 'Vic0try0na', 'played': [116, 123, 128, 130, 132, 134, 135, 139, 144, 150, 151, 154, 160, 171, 172, 173, 178, 181, 182, 185, 191, 198, 203, 210, 217, 226, 232, 234, 239, 244, 248, 253, 254, 256], 'won': [172, 198, 210, 244, 254], 'apps': 399, 'lanApps': 35, 'fncsApps': 23, 'wins': {'global': 0, 'fncs': 3, 'lan': 3, 'major': 5}, 'winRegions': ['Europe'], 'winYears': [2023, 2024, 2025, 2026]}, {'id': 'Kami', 'played': [45, 52, 59, 65, 73, 86, 93, 101, 107, 116, 123, 128, 130, 132, 134, 135, 139, 144, 145, 149, 150, 154, 160, 164, 168, 171, 172, 173, 174, 178, 181, 185, 191, 198, 203, 210, 217, 222, 226, 232, 234, 235, 239, 244, 248, 254, 256], 'won': [101, 144, 174, 234, 235], 'apps': 614, 'lanApps': 33, 'fncsApps': 33, 'wins': {'global': 0, 'fncs': 2, 'lan': 4, 'major': 5}, 'winRegions': ['Europe', 'Middle East', 'North America'], 'winYears':

In [4]:
def year_of(t):
    """Calendar year, or None when the export's date is not one."""
    m = re.match(r'^(\d{4})-', str(t.get('startdate') or ''))
    return int(m.group(1)) if m else None

bad = [t['name'] for t in tournaments if year_of(t) is None]
print(f'{len(bad)} tournaments have an unusable startdate, e.g. {bad[:3]}')

3 tournaments have an unusable startdate, e.g. ['Baron B-League August #1', 'C7S3: Console Solo Victory Cup (ZB) - Week 7: Europe', 'Adopt-A-Pro']


In [6]:
OUT = f'{BASE}/rankings.json'
YEARS = list(range(2018, 2027))
SLOTS = 10

boards = []
def board(bid, group, title, entity, tie, ranked, fmt):
    """`ranked` is [(key, label, value)] already sorted best-first."""
    if len(ranked) < SLOTS + 1:
        return
    rows = [{'key': k, 'label': l, 'value': round(v, 2), 'display': fmt(v)}
            for k, l, v in ranked[:SLOTS]]
    k, l, v = ranked[SLOTS]
    boards.append({'id': bid, 'group': group, 'title': title, 'entity': entity,
                   'tieRule': tie, 'rows': rows,
                   'next': {'key': k, 'label': l, 'value': round(v, 2)}})

money = lambda v: '${:,.0f}'.format(v)
count = lambda word: (lambda v: f'{v:,.0f} {word}' + ('' if v == 1 else 's'))

def rank(d, label_of):
    return sorted(((k, label_of(k), v) for k, v in d.items() if v > 0),
                  key=lambda e: (-e[2], e[1].lower()))

NAME = lambda page: player_of[page]['id'] if page in player_of else page
TIE_MONEY = 'Straight prize-money order. Exact ties are split by name.'
TIE_COUNT = 'Players level on the count are ranked by career earnings.'

# ---------------------------------------------------------- player: totals --
earn      = Counter()   # from placements, so it can be sliced
lan_earn  = Counter(); fncs_earn = Counter(); wc_earn = Counter()
lan_apps  = Counter(); fncs_apps = Counter()
year_earn = defaultdict(Counter)
org_earn  = Counter(); org_year  = defaultdict(Counter); org_majors = Counter()

for t, r, m, pages in PLACED:
    # `yr` is None for the handful of rows with a malformed date. They still
    # count towards career totals — the money was won — and are simply left out
    # of the per-year boards, because there is no year to file them under.
    n, yr = t['name'], year_of(t)
    for page, team in pages:
        earn[page] += m
        if yr is not None:
            year_earn[yr][page] += m
        if n in LANS:
            lan_earn[page] += m; lan_apps[page] += 1
        if n in FNCS:
            fncs_earn[page] += m; fncs_apps[page] += 1
        if n in WCUP:
            wc_earn[page] += m
        if team and team.lower() not in ('free agent', '', 'none'):
            org_earn[team] += m
            if yr is not None:
                org_year[yr][team] += m
            if n in MAJORS and r == 1:
                org_majors[team] += 1

# Career earnings uses the published column, not the sum of placements — it is
# the number Liquipedia shows on the player page and the one people remember.
career = Counter({p['pagename']: float(p['earnings'] or 0) for p in PLAYABLE})

board('career-earnings', 'Players', 'Top 10 by career earnings', 'player',
      TIE_MONEY, rank(career, NAME), money)
board('lan-earnings', 'Players', 'Top 10 by LAN earnings', 'player',
      TIE_MONEY, rank(lan_earn, NAME), money)
board('fncs-earnings', 'Players', 'Top 10 by FNCS earnings', 'player',
      TIE_MONEY, rank(fncs_earn, NAME), money)
board('earnings-no-wc', 'Players', 'Top 10 by earnings excluding the World Cup',
      'player', TIE_MONEY,
      rank(Counter({k: v - wc_earn[k] for k, v in earn.items()}), NAME), money)
board('lan-apps', 'Players', 'Top 10 by LAN appearances', 'player', TIE_COUNT,
      rank(lan_apps, NAME), count('LAN'))
board('fncs-apps', 'Players', 'Top 10 by FNCS Finals appearances', 'player',
      TIE_COUNT, rank(fncs_apps, NAME), count('final'))
board('fncs-wins', 'Players', 'Top 10 by FNCS wins', 'player', TIE_COUNT,
      rank(Counter({p['pagename']: p['fncs_wins'] for p in PLAYABLE}), NAME),
      count('title'))

major_wins = Counter()
for t, r, m, pages in PLACED:
    if t['name'] in MAJORS and r == 1:
        for page, _ in pages:
            major_wins[page] += 1
board('major-wins', 'Players', 'Top 10 by major tournament wins', 'player',
      TIE_COUNT, rank(major_wins, NAME), count('win'))

for y in YEARS:
    board(f'year-earnings:{y}', 'Players', f'Top 10 earners in {y}', 'player',
          TIE_MONEY, rank(year_earn[y], NAME), money)

# ------------------------------------------------- player: region / country --
region_of_page  = {p['pagename']: p.get('region') for p in PLAYABLE}
country_of_page = {p['pagename']: (p.get('nationalities') or [None])[0] for p in PLAYABLE}
REGIONS = sorted({r for r in region_of_page.values() if r})

for reg in REGIONS:
    sub = Counter({k: v for k, v in career.items() if region_of_page.get(k) == reg})
    board(f'region-earnings:{reg}', 'By region',
          f'Top 10 career earnings — {reg}', 'player', TIE_MONEY, rank(sub, NAME), money)
    for y in YEARS:
        sub = Counter({k: v for k, v in year_earn[y].items() if region_of_page.get(k) == reg})
        board(f'region-year-earnings:{reg}:{y}', 'By region',
              f'Top 10 earners in {y} — {reg}', 'player', TIE_MONEY, rank(sub, NAME), money)

# Only countries deep enough to field a real top ten.
by_country = defaultdict(Counter)
for page, v in career.items():
    c = country_of_page.get(page)
    if c:
        by_country[c][page] = v
for c, sub in by_country.items():
    board(f'country-earnings:{c}', 'By country',
          f'Top 10 career earnings — {c}', 'player', TIE_MONEY, rank(sub, NAME), money)

# --------------------------------------------------------------- teammates --
# Anchors people would actually recognise: the biggest earners.
mates = defaultdict(Counter)
for t, r, m, pages in PLACED:
    ids = sorted({pg for pg, _ in pages})
    if len(ids) > 1:
        for a, b in itertools.combinations(ids, 2):
            mates[a][b] += 1; mates[b][a] += 1
anchors = [p for p, _ in career.most_common(60)]
for a in anchors:
    board(f'teammates:{a}', 'Teammates',
          f'Top 10 most frequent teammates of {NAME(a)}', 'player',
          'Ranked by tournaments entered together.',
          rank(mates[a], NAME), count('tournament'))

# ----------------------------------------------------------- one tournament --
# Offline events only, per your note: globals, World Cup, DreamHack, EWC.
for n in sorted(LANS):
    t = tour_of[n]
    finishers = {}
    for tt, r, m, pages in PLACED:
        if tt['name'] != n or r is None:
            continue
        for page, _ in pages:
            finishers[page] = min(finishers.get(page, 10**9), r)
    ranked = sorted(((k, NAME(k), v) for k, v in finishers.items()),
                    key=lambda e: (e[2], e[1].lower()))
    if len(ranked) < SLOTS + 1 or ranked[SLOTS - 1][2] == ranked[SLOTS][2]:
        continue                      # 10th and 11th tied — no single right answer
    boards.append({
        'id': f'tournament:{n}', 'group': 'Tournaments',
        'title': f'Top 10 at {n}', 'entity': 'player',
        'tieRule': 'Ranked by finishing position at this event.',
        'rows': [{'key': k, 'label': l, 'value': v, 'display': f'{v}'}
                 for k, l, v in ranked[:SLOTS]],
        'next': {'key': ranked[SLOTS][0], 'label': ranked[SLOTS][1], 'value': ranked[SLOTS][2]},
        'lowerIsBetter': True,
    })

# ---------------------------------------------------------------- orgs --
org_name = {o['id']: o['name'] for o in orgs_payload['orgs']}
ORG = lambda k: org_name.get(k, k)
board('org-earnings', 'Organisations', 'Top 10 organisations by total earnings',
      'org', TIE_MONEY, rank(org_earn, ORG), money)
board('org-majors', 'Organisations', 'Top 10 organisations by major wins',
      'org', 'Ranked by wins at Epic-run majors.', rank(org_majors, ORG), count('win'))
for y in YEARS:
    board(f'org-year-earnings:{y}', 'Organisations',
          f'Top 10 organisations by earnings in {y}', 'org', TIE_MONEY,
          rank(org_year[y], ORG), money)

# ------------------------------------------------------------- countries --
CT = lambda c: c
country_total = Counter()
country_year  = defaultdict(Counter)
country_fncs  = Counter()
for p in PLAYABLE:
    c = (p.get('nationalities') or [None])[0]
    if not c:
        continue
    country_total[c] += float(p['earnings'] or 0)
    country_fncs[c]  += p['fncs_wins']
    for y in YEARS:
        country_year[y][c] += float(p.get(f'earnings_{y}') or 0)

board('country-total-earnings', 'Countries', 'Top 10 countries by player earnings',
      'country', TIE_MONEY, rank(country_total, CT), money)
board('country-fncs-wins', 'Countries', 'Top 10 countries by FNCS wins',
      'country', 'Every FNCS title won by a player of that nationality.',
      rank(country_fncs, CT), count('title'))
for y in YEARS:
    board(f'country-year-earnings:{y}', 'Countries',
          f'Top 10 countries by earnings in {y}', 'country', TIE_MONEY,
          rank(country_year[y], CT), money)
for reg in REGIONS:
    sub = Counter()
    for p in PLAYABLE:
        c = (p.get('nationalities') or [None])[0]
        if c and p.get('region') == reg:
            sub[c] += float(p['earnings'] or 0)
    board(f'region-country-earnings:{reg}', 'Countries',
          f'Top 10 countries by earnings — {reg}', 'country', TIE_MONEY,
          rank(sub, CT), money)

payload = {'generated': TODAY, 'slots': SLOTS, 'boards': boards}
with open(OUT, 'w', encoding='utf-8') as fh:
    json.dump(payload, fh, ensure_ascii=False, separators=(',', ':'))

print(f'{len(boards)} boards, {os.path.getsize(OUT)/1e6:.2f} MB')
print(Counter(b['group'] for b in boards))
for bid in ('career-earnings', 'lan-earnings', 'fncs-apps', 'org-earnings',
            'country-total-earnings'):
    b = next((x for x in boards if x['id'] == bid), None)
    print(f"\n{bid}:" if b else f"\n{bid}: MISSING")
    if b:
        for row in b['rows'][:5]:
            print(f"   {row['label']:<22} {row['display']}")

270 boards, 0.26 MB
Counter({'Tournaments': 62, 'By region': 60, 'Teammates': 58, 'By country': 47, 'Players': 17, 'Countries': 15, 'Organisations': 11})

career-earnings:
   Bugha                  $3,844,147
   aqua                   $2,201,771
   EpikWhale              $2,002,846
   psalm                  $1,880,342
   Kami                   $1,857,727

lan-earnings:
   Bugha                  $3,064,367
   psalm                  $1,834,842
   aqua                   $1,569,440
   nyhrox                 $1,515,278
   EpikWhale              $1,505,505

fncs-apps:
   EpikWhale              36 finals
   Khanada                35 finals
   Ajerss                 33 finals
   Clix                   33 finals
   Kami                   33 finals

org-earnings:
   FaZe Clan              $4,627,162
   NRG                    $4,592,003
   Sentinels              $4,114,158
   COOLER Esport          $3,741,364
   100 Thieves            $3,714,504

country-total-earnings:
   United States          

In [7]:
OUT = f'{BASE}/pools.json'

WANTED = [
    ('globals-2026', 'FNCS 2026 Globals', 'FNCS 2026  Global Championship',
     'The field for the Global Championship in Europe, 26–27 September 2026.'),
    ('ewc-2026', 'EWC 2026', 'Reload Elite Series 2026 - Championship',
     'The field for the Esports World Cup Fortnite event, August 2026.'),
]

pools = []
for pid, label, event, blurb in WANTED:
    t = tour_of.get(event)
    if not t:
        print(f'!! {event!r} not in tournaments.json — skipped')
        continue
    raw, named = set(), set()
    for row in placements:
        if row.get('tournament') != event:
            continue
        for part in (row.get('participants') or []):
            name = part.get('player') or ''
            if not name:
                continue
            raw.add(name)
            page = resolve(name)
            if page:
                named.add(page)
    pools.append({'id': pid, 'label': label, 'blurb': blurb,
                  'event': event, 'date': str(t['startdate'])[:10],
                  'players': sorted(named)})
    print(f'{label:<20} {len(raw):>3} entrants, {len(named):>3} playable')

payload = {'generated': TODAY, 'pools': pools}
with open(OUT, 'w', encoding='utf-8') as fh:
    json.dump(payload, fh, ensure_ascii=False, separators=(',', ':'))
print('\nwrote', OUT)

FNCS 2026 Globals    101 entrants, 101 playable
EWC 2026              80 entrants,  80 playable

wrote liquipedia_data/clean_data/fortnite/pools.json
